1. Preparação do Dados

Importando Base de Dados - Cadastro Nacional de Estabelecimentos de Saúde (CNES)



In [1]:
!pip install basedosdados -q

from google.colab import auth
auth.authenticate_user()

import basedosdados as bd

PROJETO = "projeto-regressao"

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# ver colunas da tabela de estabelecimentos
df_preview = bd.read_sql(
    "SELECT * FROM `basedosdados.br_ms_cnes.estabelecimento` LIMIT 5",
    billing_project_id=PROJETO
)
df_preview.columns.tolist()

Downloading: 100%|██████████|


['ano',
 'mes',
 'sigla_uf',
 'ano_atualizacao',
 'mes_atualizacao',
 'id_municipio',
 'id_municipio_6',
 'id_regiao_saude',
 'id_microrregiao_saude',
 'id_distrito_sanitario',
 'id_distrito_administrativo',
 'cep',
 'id_estabelecimento_cnes',
 'tipo_pessoa',
 'cpf_cnpj',
 'tipo_grau_dependencia',
 'cnpj_mantenedora',
 'tipo_retencao_tributos_mantenedora',
 'indicador_vinculo_sus',
 'tipo_gestao',
 'tipo_esfera_administrativa',
 'tipo_retencao_tributos',
 'tipo_atividade_ensino_pesquisa',
 'tipo_natureza_administrativa',
 'id_natureza_juridica',
 'tipo_fluxo_atendimento',
 'tipo_unidade',
 'tipo_turno',
 'tipo_nivel_hierarquia',
 'tipo_prestador',
 'banco',
 'agencia',
 'conta_corrente',
 'id_contrato_municipio_sus',
 'data_publicacao_contrato_municipal',
 'data_publicacao_contrato_estadual',
 'id_contrato_estado_sus',
 'numero_alvara',
 'data_expedicao_alvara',
 'tipo_orgao_expedidor',
 'tipo_avaliacao_acreditacao_hospitalar',
 'tipo_classificacao_acreditacao_hospitalar',
 'ano_avalia

In [ ]:
# ver colunas da tabela de profissionais
df_prof_preview = bd.read_sql(
    "SELECT * FROM `basedosdados.br_ms_cnes.profissional` LIMIT 5",
    billing_project_id=PROJETO
)
df_prof_preview.columns.tolist()

Downloading: 100%|██████████|


['ano',
 'mes',
 'sigla_uf',
 'id_municipio',
 'id_estabelecimento_cnes',
 'id_municipio_6_residencia',
 'nome',
 'tipo_vinculo',
 'id_registro_conselho',
 'tipo_conselho',
 'cartao_nacional_saude',
 'cbo_2002_original',
 'cbo_2002',
 'cbo_1994',
 'indicador_estabelecimento_terceiro',
 'indicador_vinculo_contratado_sus',
 'indicador_vinculo_autonomo_sus',
 'indicador_vinculo_outros',
 'indicador_atende_sus',
 'indicador_atende_nao_sus',
 'carga_horaria_outros',
 'carga_horaria_hospitalar',
 'carga_horaria_ambulatorial']

In [ ]:
query = """
WITH prof_count AS (
  SELECT
    id_estabelecimento_cnes,
    COUNT(DISTINCT cartao_nacional_saude) AS qtd_profissionais
  FROM `basedosdados.br_ms_cnes.profissional`
  WHERE ano = 2026 AND mes = 1
  GROUP BY id_estabelecimento_cnes
)

SELECT
  e.*,
  p.qtd_profissionais
FROM `basedosdados.br_ms_cnes.estabelecimento` e
JOIN prof_count p
  ON e.id_estabelecimento_cnes = p.id_estabelecimento_cnes
WHERE e.ano = 2026 AND e.mes = 1
"""

df = bd.read_sql(query, billing_project_id=PROJETO)
print(df.shape)
df.head()

Downloading: 100%|██████████|
(472170, 205)


,ano,mes,sigla_uf,ano_atualizacao,mes_atualizacao,id_municipio,id_municipio_6,id_regiao_saude,id_microrregiao_saude,id_distrito_sanitario,...,indicador_atendimento_vigilancia_plano_seguro_terceiro,indicador_atendimento_vigilancia_plano_saude_publico,indicador_atendimento_vigilancia_plano_saude_privado,indicador_atendimento_regulacao_sus,indicador_atendimento_regulacao_privado,indicador_atendimento_regulacao_plano_seguro_proprio,indicador_atendimento_regulacao_plano_seguro_terceiro,indicador_atendimento_regulacao_plano_saude_publico,indicador_atendimento_regulacao_plano_saude_privado,qtd_profissionais
0,2026,1,AP,2025,4,1600055,160005,1,nan,nan,...,0,0,0,0,0,0,0,0,0,3
1,2026,1,AP,2024,7,1600105,160010,nan,nan,nan,...,0,0,0,0,0,0,0,0,0,40
2,2026,1,AP,2024,12,1600238,160023,nan,nan,nan,...,0,0,0,0,0,0,0,0,0,1
3,2026,1,AP,2026,1,1600303,160030,nan,nan,nan,...,0,0,0,0,0,0,0,0,0,3
4,2026,1,AP,2022,3,1600303,160030,nan,nan,nan,...,0,0,0,0,0,0,0,0,0,4


Limpeza dos Dados e Classificação quanto a Dados Nulos

In [ ]:
colunas_para_remover = [
    # período (constante, já que filtramos um único mês)
    'ano', 'mes', 'ano_atualizacao', 'mes_atualizacao',

    # identificadores geográficos muito granulares/redundantes
    # (mantemos 'sigla_uf' como proxy geográfico mais simples)
    'id_municipio', 'id_municipio_6',
    'id_regiao_saude', 'id_microrregiao_saude',
    'id_distrito_sanitario', 'id_distrito_administrativo',

    # endereço / identificação do estabelecimento
    'cep', 'id_estabelecimento_cnes', 'cpf_cnpj', 'cnpj_mantenedora',

    # dados bancários (não descrevem a capacidade do estabelecimento)
    'banco', 'agencia', 'conta_corrente',

    # identificadores/datas de contrato administrativo
    'id_contrato_municipio_sus', 'id_contrato_estado_sus',
    'data_publicacao_contrato_municipal', 'data_publicacao_contrato_estadual',

    # licenciamento (identificadores administrativos)
    'numero_alvara', 'data_expedicao_alvara', 'tipo_orgao_expedidor',

    # datas de avaliação de outro programa (PNASS), pouco relacionadas ao alvo
    'ano_avaliacao_pnass', 'mes_avaliacao_pnass',
]

df_limpo = df.drop(columns=colunas_para_remover, errors='ignore')

print(f"Colunas antes: {df.shape[1]}")
print(f"Colunas depois: {df_limpo.shape[1]}")

Colunas antes: 205
Colunas depois: 179


In [ ]:
# quantidade e % de nulos por coluna, ordenado do pior pro melhor
import pandas as pd

nulos = df_limpo.isnull().sum()
nulos_pct = (nulos / len(df_limpo) * 100).round(2)

resumo_nulos = pd.DataFrame({
    'qtd_nulos': nulos,
    'pct_nulos': nulos_pct
}).sort_values('pct_nulos', ascending=False)

resumo_nulos[resumo_nulos['qtd_nulos'] > 0]

,qtd_nulos,pct_nulos


In [ ]:
#removendo as colunas que os dados estão 100% nulos (como mostrado na tabela acima)
colunas_100_nulas = [
    'tipo_natureza_administrativa',
    'tipo_retencao_tributos',
    'tipo_retencao_tributos_mantenedora',
    'tipo_avaliacao_acreditacao_hospitalar',
    'tipo_classificacao_acreditacao_hospitalar',
]
df_limpo = df_limpo.drop(columns=colunas_100_nulas, errors='ignore')

In [ ]:
#tipo_fluxo_atendimento (1.40%) e tipo_turno (0.28%) de dados nulos, estão rotula-los como 'não informado'

df_limpo['tipo_fluxo_atendimento'] = df_limpo['tipo_fluxo_atendimento'].fillna('nao_informado')
df_limpo['tipo_turno'] = df_limpo['tipo_turno'].fillna('nao_informado')

Separar o Dataset do Vetor Alvo a ser calculado

In [ ]:

# 1. Separar X (todas as features - alvo) e y (vetor alvo: qt de profissionais)
y = df_limpo['qtd_profissionais']
X = df_limpo.drop(columns=['qtd_profissionais'])

print("X:", X.shape)
print("y:", y.shape)

X: (472170, 173)
y: (472170,)


Separar Treino x Test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(len(X_train), "treino +", len(X_test), "teste")

377736 treino + 94434 teste


Codificar as colunas categóricas - já que não são aceitas textos em x_train e y_train

In [ ]:
# colunas categóricas (texto) vs numéricas
colunas_categoricas = X_train.select_dtypes(include='object').columns.tolist()
colunas_numericas = X_train.select_dtypes(exclude='object').columns.tolist()

print("Categóricas:", colunas_categoricas)
print("Numéricas:", len(colunas_numericas), "colunas")

Categóricas: ['sigla_uf', 'tipo_pessoa', 'tipo_grau_dependencia', 'tipo_gestao', 'tipo_esfera_administrativa', 'tipo_atividade_ensino_pesquisa', 'id_natureza_juridica', 'tipo_fluxo_atendimento', 'tipo_unidade', 'tipo_turno']
Numéricas: 161 colunas


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_cat = encoder.fit_transform(X_train[colunas_categoricas])
X_test_cat = encoder.transform(X_test[colunas_categoricas])

print(X_train_cat.shape, X_test_cat.shape)

(377736, 146) (94434, 146)


Escalonar as colunas numéricas

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train[colunas_numericas])
X_test_num = scaler.transform(X_test[colunas_numericas])

print(X_train_num.shape, X_test_num.shape)

(377736, 161) (94434, 161)


Juntar tudo de volta num único array

In [ ]:
import numpy as np

X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final = np.hstack([X_test_num, X_test_cat])

print("X_train_final:", X_train_final.shape)
print("X_test_final:", X_test_final.shape)

X_train_final: (377736, 307)
X_test_final: (94434, 307)


In [ ]:
colunas_constantes = ['tipo_nivel_hierarquia', 'tipo_prestador']

X_train = X_train.drop(columns=colunas_constantes)
X_test = X_test.drop(columns=colunas_constantes)

colunas_categoricas = [c for c in colunas_categoricas if c not in colunas_constantes]

print(colunas_categoricas)

['sigla_uf', 'tipo_pessoa', 'tipo_grau_dependencia', 'tipo_gestao', 'tipo_esfera_administrativa', 'tipo_atividade_ensino_pesquisa', 'id_natureza_juridica', 'tipo_fluxo_atendimento', 'tipo_unidade', 'tipo_turno']


Re-gerar o encoding e o escalonamento (porque removi as colunas constantes)

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np

colunas_numericas = X_train.select_dtypes(exclude='object').columns.tolist()
colunas_categoricas = X_train.select_dtypes(include='object').columns.tolist()

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_cat = encoder.fit_transform(X_train[colunas_categoricas])
X_test_cat = encoder.transform(X_test[colunas_categoricas])

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[colunas_numericas])
X_test_num = scaler.transform(X_test[colunas_numericas])

X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final = np.hstack([X_test_num, X_test_cat])

print(X_train_final.shape, X_test_final.shape)

(377736, 307) (94434, 307)


2. Análise de Correlação: cada atributo vs. o alvo